In [37]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = load_breast_cancer(as_frame=True).frame
print(df.shape)
df.head()

(569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [38]:
X = df.drop("target", axis=1)   # patient measurements
y = df["target"]                # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [39]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
}

results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    results[name] = accuracy_score(y_test, pred)
    print(name, "->", round(results[name], 4))

Logistic Regression -> 0.9825
SVM -> 0.9825
Random Forest -> 0.9561
XGBoost -> 0.9561


In [40]:
best = max(results, key=results.get)
print("Best model:", best)
pred = models[best].predict(X_test_s)
print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))

Best model: Logistic Regression
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

[[41  1]
 [ 1 71]]


In [41]:
import numpy as np

def run_all(df, target):
    X = pd.get_dummies(df.drop(target, axis=1), drop_first=True)
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "SVM": SVC(),
        "Random Forest": RandomForestClassifier(random_state=42),
        "XGBoost": XGBClassifier(eval_metric="logloss"),
    }
    results = {}
    for name, model in models.items():
        model.fit(X_train_s, y_train)
        results[name] = accuracy_score(y_test, model.predict(X_test_s))

    print(pd.Series(results).round(4))
    best = max(results, key=results.get)
    print("\nBest model:", best)
    pred = models[best].predict(X_test_s)
    print(classification_report(y_test, pred))
    print(confusion_matrix(y_test, pred))
    return results

In [42]:
df = pd.read_csv("heart.csv")
print(df.columns)       # find the target column name
print(df.isnull().sum())
heart_results = run_all(df, "target")   # change "target" if yours is named differently, e.g. "HeartDisease"

Index(['age', 'sex', 'chest_pain_type', 'resting_blood_pressure',
       'cholestoral', 'fasting_blood_sugar', 'rest_ecg', 'Max_heart_rate',
       'exercise_induced_angina', 'oldpeak', 'slope',
       'vessels_colored_by_flourosopy', 'thalassemia', 'target'],
      dtype='object')
age                              0
sex                              0
chest_pain_type                  0
resting_blood_pressure           0
cholestoral                      0
fasting_blood_sugar              0
rest_ecg                         0
Max_heart_rate                   0
exercise_induced_angina          0
oldpeak                          0
slope                            0
vessels_colored_by_flourosopy    0
thalassemia                      0
target                           0
dtype: int64
Logistic Regression    0.8732
SVM                    0.9512
Random Forest          1.0000
XGBoost                1.0000
dtype: float64

Best model: Random Forest
              precision    recall  f1-score   suppor

In [43]:
df = pd.read_csv("diabetes.csv")
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols] = df[cols].replace(0, np.nan)        # zeros here really mean "missing"
df[cols] = df[cols].fillna(df[cols].median())
diabetes_results = run_all(df, "Outcome")

Logistic Regression    0.7078
SVM                    0.7403
Random Forest          0.7792
XGBoost                0.7727
dtype: float64

Best model: Random Forest
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       100
           1       0.72      0.61      0.66        54

    accuracy                           0.78       154
   macro avg       0.76      0.74      0.75       154
weighted avg       0.77      0.78      0.77       154

[[87 13]
 [21 33]]


In [44]:
bc_results = run_all(load_breast_cancer(as_frame=True).frame, "target")

Logistic Regression    0.9825
SVM                    0.9825
Random Forest          0.9561
XGBoost                0.9561
dtype: float64

Best model: Logistic Regression
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

[[41  1]
 [ 1 71]]


In [45]:
pd.DataFrame({"Heart": heart_results, "Diabetes": diabetes_results, "Breast Cancer": bc_results}).round(4)

,Heart,Diabetes,Breast Cancer
Logistic Regression,0.8732,0.7078,0.9825
SVM,0.9512,0.7403,0.9825
Random Forest,1.0000,0.7792,0.9561
XGBoost,1.0000,0.7727,0.9561


In [46]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [47]:
df = pd.read_csv("heart.csv")
print(df.columns)
print(df.isnull().sum())
df.head()

Index(['age', 'sex', 'chest_pain_type', 'resting_blood_pressure',
       'cholestoral', 'fasting_blood_sugar', 'rest_ecg', 'Max_heart_rate',
       'exercise_induced_angina', 'oldpeak', 'slope',
       'vessels_colored_by_flourosopy', 'thalassemia', 'target'],
      dtype='object')
age                              0
sex                              0
chest_pain_type                  0
resting_blood_pressure           0
cholestoral                      0
fasting_blood_sugar              0
rest_ecg                         0
Max_heart_rate                   0
exercise_induced_angina          0
oldpeak                          0
slope                            0
vessels_colored_by_flourosopy    0
thalassemia                      0
target                           0
dtype: int64


,age,sex,chest_pain_type,resting_blood_pressure,cholestoral,fasting_blood_sugar,rest_ecg,Max_heart_rate,exercise_induced_angina,oldpeak,slope,vessels_colored_by_flourosopy,thalassemia,target
0,52,Male,Typical angina,125,212,Lower than 120 mg/ml,ST-T wave abnormality,168,No,1.0,Downsloping,Two,Reversable Defect,0
1,53,Male,Typical angina,140,203,Greater than 120 mg/ml,Normal,155,Yes,3.1,Upsloping,Zero,Reversable Defect,0
2,70,Male,Typical angina,145,174,Lower than 120 mg/ml,ST-T wave abnormality,125,Yes,2.6,Upsloping,Zero,Reversable Defect,0
3,61,Male,Typical angina,148,203,Lower than 120 mg/ml,ST-T wave abnormality,161,No,0.0,Downsloping,One,Reversable Defect,0
4,62,Female,Typical angina,138,294,Greater than 120 mg/ml,ST-T wave abnormality,106,No,1.9,Flat,Three,Fixed Defect,0


In [48]:
heart_results = run_all(df, "target")

Logistic Regression    0.8732
SVM                    0.9512
Random Forest          1.0000
XGBoost                1.0000
dtype: float64

Best model: Random Forest
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       100
           1       1.00      1.00      1.00       105

    accuracy                           1.00       205
   macro avg       1.00      1.00      1.00       205
weighted avg       1.00      1.00      1.00       205

[[100   0]
 [  0 105]]


In [49]:
df = pd.read_csv("heart.csv")
print(df.shape)
print("Duplicate rows:", df.duplicated().sum())

(1025, 14)
Duplicate rows: 723


In [50]:
df = df.drop_duplicates()
print(df.shape)
heart_results = run_all(df, "target")

(302, 14)
Logistic Regression    0.8361
SVM                    0.7705
Random Forest          0.7705
XGBoost                0.7049
dtype: float64

Best model: Logistic Regression
              precision    recall  f1-score   support

           0       0.82      0.82      0.82        28
           1       0.85      0.85      0.85        33

    accuracy                           0.84        61
   macro avg       0.83      0.83      0.83        61
weighted avg       0.84      0.84      0.84        61

[[23  5]
 [ 5 28]]


In [51]:
df = pd.read_csv("diabetes.csv")
print(df.shape)
print(df.columns)
print("Duplicates:", df.duplicated().sum())
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols] = df[cols].replace(0, np.nan)
df[cols] = df[cols].fillna(df[cols].median())
diabetes_results = run_all(df, "Outcome")

(768, 9)
Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')
Duplicates: 0
Logistic Regression    0.7078
SVM                    0.7403
Random Forest          0.7792
XGBoost                0.7727
dtype: float64

Best model: Random Forest
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       100
           1       0.72      0.61      0.66        54

    accuracy                           0.78       154
   macro avg       0.76      0.74      0.75       154
weighted avg       0.77      0.78      0.77       154

[[87 13]
 [21 33]]


In [52]:
bc_results = run_all(load_breast_cancer(as_frame=True).frame, "target")

Logistic Regression    0.9825
SVM                    0.9825
Random Forest          0.9561
XGBoost                0.9561
dtype: float64

Best model: Logistic Regression
              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

[[41  1]
 [ 1 71]]


In [53]:
summary = pd.DataFrame({
    "Heart Disease": heart_results,
    "Diabetes": diabetes_results,
    "Breast Cancer": bc_results,
}).round(4)
summary

,Heart Disease,Diabetes,Breast Cancer
Logistic Regression,0.8361,0.7078,0.9825
SVM,0.7705,0.7403,0.9825
Random Forest,0.7705,0.7792,0.9561
XGBoost,0.7049,0.7727,0.9561


In [54]:
!cp /heart.csv /diabetes.csv /content/

In [55]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
